# Prompt Engineering Portfolio

This notebook demonstrates 10+ prompt engineering techniques with real examples for text classification, summarization, code generation, and data extraction. It compares GPT, Claude, and a local Ollama-based model when available.

## Setup and provider wrappers

The next cell loads environment variables and defines wrappers for OpenAI, Anthropic, and Ollama (if installed locally).

In [ ]:
import os
import json
import shutil
import subprocess
import textwrap
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
OLLAMA_INSTALLED = shutil.which('ollama') is not None

print('OpenAI key available:', bool(OPENAI_API_KEY))
print('Anthropic key available:', bool(ANTHROPIC_API_KEY))
print('Ollama installed:', OLLAMA_INSTALLED)

In [ ]:
def safe_text(response):
    if response is None:
        return ''
    if isinstance(response, str):
        return response.strip()
    try:
        if hasattr(response, 'output_text'):
            return response.output_text.strip()
        if hasattr(response, 'content') and isinstance(response.content, str):
            return response.content.strip()
        if hasattr(response, 'output'):
            pieces = []
            for item in response.output:
                if isinstance(item, dict) and item.get('type') == 'output_text':
                    for content in item.get('content', []):
                        pieces.append(content.get('text', ''))
            if pieces:
                return ''.join(pieces).strip()
    except Exception:
        pass
    try:
        return json.dumps(response, default=str, indent=2)
    except Exception:
        return str(response)

openai_client = None
anthropic_client = None

if OPENAI_API_KEY:
    try:
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
    except Exception as exc:
        print('OpenAI client import failed:', exc)

if ANTHROPIC_API_KEY:
    try:
        from anthropic import Anthropic, HUMAN_PROMPT, AI_PROMPT
        anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    except Exception as exc:
        print('Anthropic client import failed:', exc)

def call_openai(prompt, model='gpt-4o-mini', temperature=0.2):
    if not openai_client:
        return 'OpenAI client not configured. Set OPENAI_API_KEY.'
    try:
        response = openai_client.responses.create(model=model, input=prompt, temperature=temperature, max_output_tokens=400)
        return safe_text(response)
    except Exception as exc:
        return f'OpenAI call failed: {exc}'

def call_claude(prompt, model='claude-3.5', temperature=0.2):
    if not anthropic_client:
        return 'Anthropic client not configured. Set ANTHROPIC_API_KEY.'
    try:
        full_prompt = f'{HUMAN_PROMPT}{prompt}{AI_PROMPT}'
        response = anthropic_client.completions.create(model=model, prompt=full_prompt, max_tokens_to_sample=400, temperature=temperature)
        return safe_text(response.completion if hasattr(response, 'completion') else response)
    except Exception as exc:
        return f'Claude call failed: {exc}'

def call_ollama(prompt, model='llama2'):
    if not OLLAMA_INSTALLED:
        return 'Ollama not installed in this environment.'
    try:
        result = subprocess.run(['ollama', 'chat', model, '--prompt', prompt], capture_output=True, text=True, timeout=120)
        if result.returncode == 0:
            return result.stdout.strip()
        return f'Ollama error: {result.stderr.strip() or result.stdout.strip()}'
    except Exception as exc:
        return f'Ollama execution failed: {exc}'

providers = {
    'GPT': lambda prompt: call_openai(prompt, model='gpt-4o-mini'),
    'Claude': lambda prompt: call_claude(prompt, model='claude-3.5'),
    'Ollama': lambda prompt: call_ollama(prompt, model='llama2'),
}

def compare_providers(prompt, label):
    results = {provider: fn(prompt) for provider, fn in providers.items()}
    print(f'--- {label} ---')
    for provider, output in results.items():
        print(f"[{provider}]\n{output}\n")
    return results

## Technique 1: Zero-shot sentiment classification

This technique provides a simple, direct instruction to classify text without examples.

In [ ]:
zero_shot_prompt = textwrap.dedent('''
You are a sentiment classifier. For each review, return one of: Positive, Neutral, Negative.

Review 1: I loved the product, it works beautifully and arrived faster than expected.
Review 2: The app keeps crashing and customer support did not respond.
Review 3: The interface is okay, but the pricing is a little high.
''')

zero_shot_results = compare_providers(zero_shot_prompt, 'Zero-shot sentiment classification')
pd.DataFrame(zero_shot_results, index=['Sentiment']).T

## Technique 2: Few-shot classification

Provide labeled examples to guide the model toward consistent classification.

In [ ]:
few_shot_prompt = textwrap.dedent('''
Use the examples below to classify the new review as Positive, Neutral, or Negative.

Example 1: The new update improved performance and I am happy. -> Positive
Example 2: The product is fine but missing a few useful features. -> Neutral
Example 3: I am disappointed and will not buy again. -> Negative

Review: The features are powerful but the installation was frustrating.
''')

few_shot_results = compare_providers(few_shot_prompt, 'Few-shot sentiment classification')
pd.DataFrame(few_shot_results, index=['Sentiment']).T

## Technique 3: Chain-of-thought reasoning

Encourage the model to reason step-by-step before answering.

In [ ]:
cot_prompt = textwrap.dedent('''
Solve the question with step-by-step reasoning and then give the final answer.

Question: If a store sold 120 units in March and then sold 25% more in April, how many units did it sell in April?
''')

cot_results = compare_providers(cot_prompt, 'Chain-of-thought reasoning')
pd.DataFrame(cot_results, index=['Reasoning']).T

## Technique 4: Structured output for data extraction

Ask the model to return valid JSON with specific fields.

In [ ]:
invoice_text = textwrap.dedent('''
Invoice ID: 76432
Date: 2026-02-18
Vendor: Acme Widgets
Total: $1,250.00
Due: 2026-03-18
Items:
  - Widget A x3 @ $250.00
  - Widget B x1 @ $500.00
''')

structured_prompt = textwrap.dedent(f'''
Extract the invoice details from the text below and respond only with JSON.
The JSON must include: invoice_id, date, vendor, total, due_date, and items.
Each item should have: description, quantity, and price.

Text:
{invoice_text}
''')

structured_results = compare_providers(structured_prompt, 'Structured extraction')
pd.DataFrame(structured_results, index=['JSON']).T

## Technique 5: Role-playing summarization

Ask the model to summarize from the perspective of a role, such as a product manager.

In [ ]:
meeting_notes = textwrap.dedent('''
The customer wants a faster onboarding flow, clearer error messages, and better reporting dashboards.
Development is on track for mobile release, but team bandwidth is tight.
We need to prioritize stability before adding new integrations.
''')

role_play_prompt = textwrap.dedent(f'''
You are a product manager writing an executive summary.
Summarize the meeting notes below, focusing on customer needs, risks, and the next priority.

Meeting notes:
{meeting_notes}
''')

role_play_results = compare_providers(role_play_prompt, 'Role-playing summarization')
pd.DataFrame(role_play_results, index=['Summary']).T

## Technique 6: Code generation with constraints

Request a function with clear behavior and formatting requirements.

In [ ]:
code_prompt = textwrap.dedent('''
Write a Python function named `validate_email` that checks whether an email address is valid.
The function should use a regular expression, return True/False, and include a concise docstring.
Do not include any example usage or explanation.
''')

code_results = compare_providers(code_prompt, 'Code generation')
pd.DataFrame(code_results, index=['Code']).T

## Technique 7: Prompt decomposition

Break a larger task into smaller subtasks before answering.

In [ ]:
decomposition_prompt = textwrap.dedent('''
You are designing a product launch checklist.
First list the key launch phases. Then, for each phase, provide the most important task.
Return the result as a numbered list with phase and task.
''')

decomposition_results = compare_providers(decomposition_prompt, 'Prompt decomposition')
pd.DataFrame(decomposition_results, index=['Checklist']).T

## Technique 8: Self-critique and improvement

Ask the model to analyze its own initial response and improve it.

In [ ]:
self_critique_prompt = textwrap.dedent('''
Write a short executive summary of the product launch risks in three sentences.
Then critique the summary and rewrite it to be more specific and actionable.
''')

self_critique_results = compare_providers(self_critique_prompt, 'Self-critique')
pd.DataFrame(self_critique_results, index=['Summary']).T

## Technique 9: Structured output comparison for evaluation

Create a quick summary table of which providers were available and how they responded to the portfolio prompts.

In [ ]:
availability = {
    'Provider': ['GPT', 'Claude', 'Ollama'],
    'Available': [bool(openai_client), bool(anthropic_client), OLLAMA_INSTALLED],
}
availability_df = pd.DataFrame(availability)
availability_df